# GT to Markov Network Workflow

This notebook uses `gt_markov_setup.py` to reproduce the reusable loading and feature-construction steps from `gt.py`, then passes the prepared tables into `markov_network.py`.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

workspace_root = Path.cwd().resolve()
while workspace_root != workspace_root.parent and not (workspace_root / "markov_network.py").exists():
    workspace_root = workspace_root.parent

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from gt_markov_setup import (
    GTColumnConfig,
    GTFeatureConfig,
    GTSubsetConfig,
    load_and_prepare_markov_network_inputs,
    prepare_estimator_input,
)
from markov_network import (
    MarkovNetworkConfig,
    MarkovNetworkEstimator,
    MarkovNetworkPermutationTester,
    PermutationTestConfig,
)

## Configure inputs

Set the CSV and parcellation paths plus the column names used by your dataframe. If your CSV already has a precomputed percent-improvement column, set `outcome_column` and leave the baseline/followup pair unused.


In [ ]:
data_path = Path("/Volumes/PdBwh/03h_ADVANCE_Alzheimer_DBS/savir_analysis/dataframes/data_cog11_v5.csv")
parcellation_path = Path("/Users/savirmadan/Downloads/aal_atlas/AAL_V7_fine_combined_parcellation.nii.gz")

columns = GTColumnConfig(
    efield_column="efield_file",
    atrophy_column="atrophy_data_v2",
    connectivity_column="t_file",
    age_column="age",
    # Use this when the CSV already contains the improvement metric you want to model.
    outcome_column="one_year_mem",  # e.g. "adas_cog11_percent_improvement"
    # These are only used when outcome_column is None.
    baseline_outcome_column=None,#"Baseline ADAS-Cog11",
    followup_outcome_column=None,#,"12 months Post-Stimulation ADAS-Cog 11",
    # Set this if global-atrophy subsetting should use a different atrophy path column.
    global_atrophy_column=None,
)

feature_config = GTFeatureConfig(
    include_age=True,
    include_atrophy=True,
    include_fc=True,
    include_interactions=True,
)

subset_config = GTSubsetConfig(
    enabled=False,
    method="global_atrophy",
    fraction=0.5,
    selection="top",
    n_subjects=None,
    random_state=42,
)

## Build the GT-derived inputs

This cell loads the dataframe, loads the parcellation, extracts network-level atrophy and FC means, builds the interaction terms from `gt.py`, and splits the result into `features`, `covariates`, and `outcome`.


In [ ]:
prepared = load_and_prepare_markov_network_inputs(
    data_path=data_path,
    parcellation_path=parcellation_path,
    columns=columns,
    feature_config=feature_config,
    subset_config=subset_config,
    outcome_name="Y",
)

print(f"Subjects used: {len(prepared.subject_table)}")
print(f"Networks: {prepared.parcellation.network_names}")
print(f"Feature table shape: {prepared.feature_table.shape}")
print(f"Features shape: {prepared.features.shape}")
print(f"Covariates shape: {None if prepared.covariates is None else prepared.covariates.shape}")
print(f"Outcome length: {len(prepared.outcome)}")

In [ ]:
prepared.feature_table.head()

## Pass the data to `markov_network.py`

`prepared.features`, `prepared.covariates`, and `prepared.outcome` are the exact objects needed by the estimator.


In [ ]:
estimator = MarkovNetworkEstimator(
    MarkovNetworkConfig(
        alpha_grid=tuple(np.logspace(-4, 0, 10)),
        edge_threshold=0.0,
        max_iter=1000,
        tolerance=1e-4,
        cv_folds=5,
        standardize=True,
        adaptive_fallback=True,
    )
)

prepared_input = prepare_estimator_input(estimator, prepared)
fit_result = estimator.fit_prepared(prepared_input)

print(f"Chosen alpha: {fit_result.lambda_value:.6f}")
print(f"Precision matrix shape: {fit_result.precision.shape}")
print(f"Partial correlation matrix shape: {fit_result.pcorr.shape}")
fit_result.edge_table.head()

## Optional permutation testing

Run this cell when you want significance estimates for the observed edges.


In [ ]:
permutation_tester = MarkovNetworkPermutationTester(
    estimator,
    PermutationTestConfig(
        n_permutations=100,
        alpha=0.05,
        random_state=42,
        n_jobs=-1,
    ),
)

permutation_results = permutation_tester.run_prepared(
    prepared_input,
    obs_res=fit_result,
)
permutation_results.head()

## Optional exports

Save the constructed subject-level table before or after fitting if you want a flat file for inspection.


In [ ]:
output_dir = workspace_root / "outputs"
output_dir.mkdir(exist_ok=True)

prepared.feature_table.to_csv(output_dir / "gt_markov_feature_table.csv", index=False)
fit_result.edge_table.to_csv(output_dir / "markov_edges.csv", index=False)